# Chicago TNP Trips — Price Dynamics Analysis

Split each fare into a rule-like **base** price and a demand-driven **variable** part,
check that the variable part behaves like surge, and track how dynamic pricing changed
from 2022 to 2024.

Data source: MatrixOne (`chicago_tnp.trips`), Chicago rideshare trips, 100k/month sample.

In [ ]:
# Install dependencies (run once)
%pip install -q pandas numpy scikit-learn matplotlib requests pymysql

## 1. Load trip-level data from MatrixOne

In [ ]:
import pymysql
import pandas as pd

conn = pymysql.connect(host="127.0.0.1", port=6001,
                       user="root", password="111", database="chicago_tnp")

# Pull trip-level rows. Drop pooled trips: their split fares distort per-trip price.
sql = """
SELECT trip_start_timestamp, trip_miles, trip_seconds,
       fare, additional_charges, trip_total,
       pickup_h3, pickup_community_area
FROM trips
WHERE shared_trip_authorized = 0
"""
df = pd.read_sql(sql, conn)
conn.close()

# DECIMAL columns come back as Python Decimal; cast to float so we can do math.
for c in ["fare", "additional_charges", "trip_total"]:
    df[c] = df[c].astype(float)

print("rows:", len(df))
df.head()

## 2. Time features

Turn the timestamp into simple, meaningful buckets (this is binning).

In [ ]:
df["trip_start_timestamp"] = pd.to_datetime(df["trip_start_timestamp"])
dt = df["trip_start_timestamp"].dt

df["year"]       = dt.year
df["month"]      = dt.month
df["hour"]       = dt.hour
df["dow"]        = dt.dayofweek            # 0=Mon ... 6=Sun
df["is_weekend"] = (dt.dayofweek >= 5).astype("int8")
df["is_night"]   = ((dt.hour >= 22) | (dt.hour < 6)).astype("int8")
df["is_rush"]    = ((dt.dayofweek < 5) &
                    (((dt.hour >= 7) & (dt.hour < 10)) | ((dt.hour >= 16) & (dt.hour < 19)))).astype("int8")
df[["year", "hour", "is_weekend", "is_night", "is_rush"]].head()

## 3. Split fare into base + variable

Idea: a fare should grow with distance and time. That predictable part is the **base**
(a rule-like price). Whatever is left over is the **variable** part (demand-driven surge).

We fit the base **separately for each year**, so inflation is absorbed into each year's
base, and the variable part is each trip's deviation from *that year's* rule price.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

df["base"] = np.nan
coefs = []
for y in sorted(df["year"].unique()):
    m = df["year"] == y
    model = LinearRegression().fit(df.loc[m, ["trip_miles", "trip_seconds"]], df.loc[m, "fare"])
    df.loc[m, "base"] = model.predict(df.loc[m, ["trip_miles", "trip_seconds"]])
    coefs.append({"year": y,
                  "start_$": round(model.intercept_, 2),
                  "per_mile_$": round(model.coef_[0], 2),
                  "per_min_$": round(model.coef_[1] * 60, 2)})  # coef is per second -> *60

df["variable"] = df["fare"] - df["base"]   # leftover = demand-driven part

# The coefficients look like a real fare table, and stay stable across years.
print(pd.DataFrame(coefs).to_string(index=False))

## 4. Check that "variable" really is demand-driven

If it is surge, it should be **positive when demand is high** (rush, night) and
**negative when demand is low** (midday).

In [ ]:
print("mean variable by rush hour:")
print(df.groupby("is_rush")["variable"].mean().round(2))
print("\nmean variable by night:")
print(df.groupby("is_night")["variable"].mean().round(2))

In [ ]:
import matplotlib.pyplot as plt

df.groupby("hour")["variable"].mean().plot(kind="bar", figsize=(10, 3),
    title="Average variable (surge) component by hour")
plt.axhline(0, color="k", lw=0.8)
plt.ylabel("$ above base"); plt.xlabel("hour"); plt.tight_layout(); plt.show()

## 5. Compare prices across years

Compare years fairly by holding distance roughly fixed (same distance bucket).
Each year has all 12 months sampled equally, so full-year vs full-year is season-balanced.

**View A: nominal median fare** per distance bucket, per year.

In [ ]:
df["mile_bin"] = pd.cut(df["trip_miles"], bins=[0, 2, 5, 10, 20, 100])
df.pivot_table(values="fare", index="mile_bin", columns="year",
               aggfunc="median", observed=True).round(2)

**View B: did the demand-driven premium change?**
Average nighttime variable per distance bucket, per year. A consistent drop across
all buckets means the change is real, not driven by one trip type.

In [ ]:
night = df[df["is_night"] == 1]
night.pivot_table(values="variable", index="mile_bin", columns="year",
                  aggfunc="mean", observed=True).round(2)

## 6. Yearly strength of dynamic pricing

Two simple per-year measures:
- **night_premium**: average extra (vs base) that night trips cost
- **volatility_std**: spread of the variable part (bigger = more price swing)

In [ ]:
strength = pd.DataFrame({
    "night_premium": df[df["is_night"] == 1].groupby("year")["variable"].mean(),
    "volatility_std": df.groupby("year")["variable"].std(),
}).round(2)
print(strength)

## 7. Demand growth vs pricing strength

The 100k/month sample erased true volume, so pull real yearly totals from the
Chicago data portal API and put them next to the pricing strength. Requires internet.

In [ ]:
import requests

def yearly_population(dataset_id):
    url = f"https://data.cityofchicago.org/resource/{dataset_id}.json"
    params = {"$select": "date_extract_y(trip_start_timestamp) AS yr, count(*) AS n",
              "$group": "yr", "$order": "yr", "$limit": 5000}
    out = pd.DataFrame(requests.get(url, params=params).json())
    return out.astype({"yr": int, "n": int})

pop = pd.concat([yearly_population("m6dm-c72p"),   # TNP trips 2018-2022
                 yearly_population("n26f-ihde")],  # TNP trips 2023-2024
                ignore_index=True)
pop = pop[pop["yr"].between(2022, 2024)].set_index("yr")
pop.index.name = "year"

# Demand growth (relative to 2022) next to the pricing-strength measures
compare = pop.join(strength)
compare["demand_vs_2022"] = (compare["n"] / compare.loc[2022, "n"]).round(2)
compare

## 8. Summary

**Finding.** From 2022 to 2024, Chicago rideshare demand grew about 32% (real trips
69M -> 91M), but dynamic pricing got *weaker*, not stronger. The average night premium
fell from about +\$2.48 per trip to +\$0.05 — consistently across every distance bucket —
and overall price volatility narrowed (std 8.4 -> 6.7). In short: more demand, flatter pricing.

**Method.** For each year, fit `fare ~ distance + time` (linear regression) to get a
rule-like base price (about \$5 start + \$0.96/mile + \$0.35/min). `variable = fare - base`
is the leftover, demand-driven part; it was validated to move with demand (positive in
rush hours and at night, negative midday). Real yearly volumes came from the city portal
API, since the monthly 100k sampling removed absolute volume.

**Limits.** We can confirm *what* happened (premium fell, demand rose), but not *why*
(supply recovery vs platform strategy) — that needs driver/supply data we don't have.
Fares are rounded to \$2.50 and the linear base underestimates long trips, so premium
magnitudes are approximate; the trend is reliable.